# DNB — Offline Smoke Tests

Interactive validation of the pipeline:

```
WaveletConvolution → TargetWaveDetector (activation) → AmplitudeMonitor (inhibition) → StimTrigger
```

All parameters are defined once in the config dict below.

**Event semantics:**
- `SLOW_WAVE` — detection event (logged always)
- `STIM` — audio stimulation. Pulse 1 fires immediately at the detected upwave; pulses 2+ at predicted future peaks.

In [ ]:
from math import pi

CFG = {
    'pipeline': {
        'sample_rate': 1000.0,
        'n_channels': 1,
        'buffer_duration': 10.0,
        'chunk_duration': 0.05,
    },
    'wavelet': {
        'freq_min': 0.5,
        'freq_max': 4.0,
        'n_freqs': 10,
        'n_cycles_base': 3.0,
    },
    'target_wave': {
        'id': 'slow_wave',
        'freq_range': (0.5, 4.0),
        'target_phase': 0.0,
        'phase_tolerance': 0.05,
        'amp_min': 1000.0,
        'amp_max': 10000.0,
        'warmup_chunks': 3,
        'amp_smoothing': 5,
    },
    'amplitude_monitor': {
        'enabled': True,
        'id': 'ied_monitor',
        'freq_range': (80.0, 120.0),
        'threshold': None,
        'adaptive_n_std': 3.0,
        'warmup_chunks': 20,
        'baseline_chunks': 100,
    },
    'trigger': {
        'activation_detector_id': 'slow_wave',
        'inhibition_detector_id': 'ied_monitor',
        'n_pulses': 1,
        'backoff_s': 5.0,
        'inhibition_cooldown_s': 5.0,
    },
    'synthetic': {
        'duration_s': 120.0,
        'snr': 5.0,
        'n_slow_waves': 15,
        'n_ieds': 40,
        'sw_amplitude': 500.0,
        'ied_amplitude': 3000.0,
        'seed': 42,
    },
    'validation': {
        'time_tolerance': 0.5,
        'snr_levels': [1.0, 2.0, 3.0, 5.0, 10.0],
    },
}

print('Config loaded.')
print(f"  chunk_duration = {CFG['pipeline']['chunk_duration']}s")
print(f"  target_phase   = {CFG['target_wave']['target_phase']:.2f} rad")
print(f"  n_pulses       = {CFG['trigger']['n_pulses']}")
print(f"  IED band       = {CFG['amplitude_monitor']['freq_range']}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from dnb import Pipeline, FileSource, PipelineConfig, EventType
from dnb.modules import (
    WaveletConvolution, TargetWaveDetector, AmplitudeMonitor, StimTrigger,
)
from dnb.validation.ground_truth import validate, Annotation
from test_data import TestData

import dnb
print(f'DNB v{dnb.__version__}')

td = TestData()

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 4)
plt.rcParams['figure.dpi'] = 100

In [ ]:
def make_pipeline_config():
    p = CFG['pipeline']
    return PipelineConfig(
        sample_rate=p['sample_rate'], n_channels=p['n_channels'],
        buffer_duration=p['buffer_duration'], chunk_duration=p['chunk_duration'],
    )

def make_wavelet():
    w = CFG['wavelet']
    return WaveletConvolution(
        freq_min=w['freq_min'], freq_max=w['freq_max'],
        n_freqs=w['n_freqs'], n_cycles_base=w['n_cycles_base'],
    )

def make_detector(**overrides):
    tw = {**CFG['target_wave'], **overrides}
    return TargetWaveDetector(
        id=tw['id'], freq_range=tw['freq_range'],
        target_phase=tw['target_phase'], phase_tolerance=tw['phase_tolerance'],
        amp_min=tw['amp_min'], amp_max=tw['amp_max'],
        warmup_chunks=tw['warmup_chunks'], amp_smoothing=tw['amp_smoothing'],
    )

def make_inhibitor(**overrides):
    am = {**CFG['amplitude_monitor'], **overrides}
    return AmplitudeMonitor(
        id=am['id'], freq_range=am['freq_range'],
        threshold=am['threshold'], adaptive_n_std=am['adaptive_n_std'],
        warmup_chunks=am['warmup_chunks'], baseline_chunks=am['baseline_chunks'],
    )

def make_trigger(**overrides):
    tr = {**CFG['trigger'], **overrides}
    return StimTrigger(
        activation_detector_id=tr['activation_detector_id'],
        inhibition_detector_id=tr['inhibition_detector_id'],
        n_pulses=tr['n_pulses'], backoff_s=tr['backoff_s'],
        inhibition_cooldown_s=tr['inhibition_cooldown_s'],
    )

def get_detections(events):
    return [e for e in events if e.event_type == EventType.SLOW_WAVE]

def get_stims(events, pulse_index=None):
    out = [e for e in events if e.event_type == EventType.STIM]
    if pulse_index is not None:
        out = [e for e in out if e.metadata.get('pulse_index') == pulse_index]
    return out

print('Helpers ready.')

---
## 1. Clean sine wave — verify phase detection

With n_pulses=1, the STIM should fire at the same time as the detection (the upwave).

In [ ]:
path = td.clean_sine()

pipeline = Pipeline(
    source=FileSource(str(path)),
    modules=[
        make_wavelet(),
        make_detector(amp_min=100.0, amp_max=100000.0, phase_tolerance=0.1),
        make_trigger(n_pulses=1, inhibition_detector_id=None, backoff_s=0.0),
    ],
    config=make_pipeline_config(),
)

events = pipeline.run_offline()
detections = get_detections(events)
stims = get_stims(events)

print(f'{len(detections)} detections, {len(stims)} stims')
if detections:
    phases = np.array([e.metadata['phase'] for e in detections])
    det_times = np.array([e.timestamp for e in detections])
    stim_times = np.array([e.timestamp for e in stims])
    target = CFG['target_wave']['target_phase']
    print(f'Detection phase: mean={np.mean(phases):.3f} std={np.std(phases):.3f} (target={target:.3f})')

    # n_pulses=1: stim fires at detection time
    if stims:
        delays = stim_times[:len(det_times)] - det_times[:len(stim_times)]
        print(f'Stim delay from detection: mean={np.mean(delays)*1000:.1f}ms (should be ~0)')

    data = np.load(str(path))
    sig = data['continuous'][0]
    t = np.arange(len(sig)) / CFG['pipeline']['sample_rate']

    fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
    axes[0].plot(t, sig, 'b-', lw=0.5, alpha=0.7)
    for e in stims:
        axes[0].axvline(e.timestamp, color='red', alpha=0.6, lw=1)
    axes[0].set_ylabel('Amplitude (µV)')
    axes[0].set_title('Clean 1 Hz sine — red=stim (fires at detected upwave)')

    axes[1].scatter(det_times, phases, c='green', s=20, zorder=5, label='detection phase')
    axes[1].axhline(target, color='green', ls='--', label=f'target={target:.2f}')
    axes[1].set_ylabel('Phase (rad)')
    axes[1].set_xlabel('Time (s)')
    axes[1].legend()
    plt.tight_layout()
    plt.show()

---
## 2. Synthetic slow waves — detection + validation

In [ ]:
syn = CFG['synthetic']
path_sw, gt_events = td.slow_waves(snr=syn['snr'])
sw_gt = [e for e in gt_events if e.metadata.get('type') == 'SW']
print(f'Planted: {len(sw_gt)} slow waves')

data = np.load(str(path_sw))
sig = data['continuous'][0]
t = np.arange(len(sig)) / CFG['pipeline']['sample_rate']

pipeline = Pipeline(
    source=FileSource(str(path_sw)),
    modules=[
        make_wavelet(),
        make_detector(),
        make_trigger(n_pulses=1, inhibition_detector_id=None, backoff_s=2.0),
    ],
    config=make_pipeline_config(),
)
all_events = pipeline.run_offline()
detections = get_detections(all_events)
stims = get_stims(all_events)
print(f'Detections: {len(detections)}, Stims: {len(stims)}')

annotations = [Annotation(timestamp=e.timestamp, channel=e.channel_id, event_type='SW') for e in sw_gt]
report = validate(detections, annotations, time_tolerance=CFG['validation']['time_tolerance'])
print(report.summary())

fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(t, sig, 'b-', lw=0.3, alpha=0.5)
for e in sw_gt:
    ax.axvline(e.timestamp, color='green', alpha=0.4, lw=2,
               label='GT' if e == sw_gt[0] else '')
for i, e in enumerate(stims):
    ax.axvline(e.timestamp, color='red', alpha=0.6, lw=1, ls='--',
               label='stim' if i == 0 else '')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')
ax.set_title('Stims (red) vs Ground Truth (green)')
ax.legend()
plt.tight_layout()
plt.show()

---
## 3. N-pulse stimulation

- `n_pulses=0` → SLOW_WAVE only, no stims
- `n_pulses=1` → 1 stim at the detected upwave (immediate)
- `n_pulses=3` → stim at detection + 2 more at next predicted peaks

In [ ]:
for n_pulses in [0, 1, 3]:
    pipe = Pipeline(
        source=FileSource(str(path_sw)),
        modules=[
            make_wavelet(),
            make_detector(),
            make_trigger(n_pulses=n_pulses, inhibition_detector_id=None, backoff_s=5.0),
        ],
        config=make_pipeline_config(),
    )
    dets = pipe.run_offline()
    sw_dets = get_detections(dets)
    all_stims = get_stims(dets)
    print(f'n_pulses={n_pulses}: {len(sw_dets)} detections, {len(all_stims)} stims')

    if n_pulses >= 1 and all_stims:
        for d in sw_dets[:2]:
            det_t = d.timestamp
            freq = d.metadata['frequency']
            period = 1.0 / freq
            my_stims = sorted(
                [s for s in all_stims if abs(s.metadata.get('detection_time', -1) - det_t) < 0.001],
                key=lambda e: e.metadata['pulse_index'],
            )
            print(f'  det t={det_t:.3f}s freq={freq:.2f}Hz:')
            for s in my_stims:
                k = s.metadata['pulse_index']
                expected = det_t + (k - 1) * period
                print(f'    stim {k}/{n_pulses} t={s.timestamp:.3f}s '
                      f'(expected={expected:.3f}s Δ={s.timestamp - det_t:.3f}s)')
    print()

In [ ]:
pipe_3 = Pipeline(
    source=FileSource(str(path_sw)),
    modules=[
        make_wavelet(),
        make_detector(),
        make_trigger(n_pulses=3, inhibition_detector_id=None, backoff_s=5.0),
    ],
    config=make_pipeline_config(),
)
dets_3 = pipe_3.run_offline()
stims_3 = get_stims(dets_3)

pulse_colors = {1: 'red', 2: 'orange', 3: 'purple'}
labels_used = set()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(t, sig, 'b-', lw=0.3, alpha=0.5)
for e in sw_gt:
    ax.axvline(e.timestamp, color='green', alpha=0.3, lw=2,
               label='GT' if e == sw_gt[0] else '')
for e in stims_3:
    pi_idx = e.metadata['pulse_index']
    c = pulse_colors.get(pi_idx, 'gray')
    lbl = f'stim {pi_idx}' if pi_idx not in labels_used else ''
    labels_used.add(pi_idx)
    ax.axvline(e.timestamp, color=c, alpha=0.7, lw=1.2, ls='--', label=lbl)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')
ax.set_title('3-pulse: stim 1 at detection, stim 2+3 at predicted peaks')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

### 3b. Verify pulse timing

Stim k should land at `detection_time + (k-1)/freq`:
- k=1 at t₀ (immediate)
- k=2 at t₀ + 1/freq
- k=3 at t₀ + 2/freq

In [ ]:
from collections import defaultdict

sequences = defaultdict(list)
for e in stims_3:
    sequences[e.metadata['detection_time']].append(e)

all_errors = []
for det_t, seq in sorted(sequences.items()):
    seq.sort(key=lambda e: e.metadata['pulse_index'])
    freq = seq[0].metadata['frequency']
    period = 1.0 / freq
    for e in seq:
        k = e.metadata['pulse_index']
        expected = det_t + (k - 1) * period
        all_errors.append(e.timestamp - expected)

errs = np.array(all_errors) * 1000
print(f'{len(sequences)} sequences, {len(all_errors)} stim events')
print(f'Timing error: {np.mean(errs):.1f} ± {np.std(errs):.1f} ms')
print(f'  (quantised to chunk_duration={CFG["pipeline"]["chunk_duration"]*1000:.0f} ms)')

---
## 4. IED inhibition

In [ ]:
path_ied, gt_ied = td.slow_waves_with_ieds(
    snr=syn['snr'], ied_amplitude=syn['ied_amplitude'],
)
sw_gt_ied = [e for e in gt_ied if e.metadata.get('type') == 'SW']
ied_gt = [e for e in gt_ied if e.metadata.get('type') == 'IED']
print(f'Planted: {len(sw_gt_ied)} SWs, {len(ied_gt)} IEDs')

data_ied = np.load(str(path_ied))
sig_ied = data_ied['continuous'][0]
t_ied = np.arange(len(sig_ied)) / CFG['pipeline']['sample_rate']

pipe_inh = Pipeline(
    source=FileSource(str(path_ied)),
    modules=[make_wavelet(), make_detector(), make_inhibitor(),
             make_trigger(n_pulses=1, backoff_s=5.0)],
    config=make_pipeline_config(),
)
stims_inh = get_stims(pipe_inh.run_offline())

pipe_no = Pipeline(
    source=FileSource(str(path_ied)),
    modules=[make_wavelet(), make_detector(),
             make_trigger(n_pulses=1, inhibition_detector_id=None, backoff_s=3.0)],
    config=make_pipeline_config(),
)
stims_no = get_stims(pipe_no.run_offline())

def near_ieds(stims, ieds, win=1.0):
    return sum(1 for s in stims if any(abs(s.timestamp - i.timestamp) < win for i in ieds))

print(f'With inhibition:    {len(stims_inh)} stims, {near_ieds(stims_inh, ied_gt)} near IEDs')
print(f'Without inhibition: {len(stims_no)} stims, {near_ieds(stims_no, ied_gt)} near IEDs')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
for ax, stims, label in [
    (axes[0], stims_no, 'WITHOUT inhibition'),
    (axes[1], stims_inh, 'WITH inhibition'),
]:
    ax.plot(t_ied, sig_ied, 'b-', lw=0.3, alpha=0.5)
    for e in sw_gt_ied:
        ax.axvline(e.timestamp, color='green', alpha=0.3, lw=2)
    for e in ied_gt:
        ax.axvspan(e.timestamp - 0.1, e.timestamp + 0.1, color='orange', alpha=0.3)
    for e in stims:
        ax.axvline(e.timestamp, color='red', alpha=0.7, lw=1, ls='--')
    ax.set_ylabel('Amp')
    ax.set_title(label)
axes[1].set_xlabel('Time (s)')
plt.tight_layout()
plt.show()

### 4b. N-pulse with IED inhibition

In [ ]:
pipe_3_inh = Pipeline(
    source=FileSource(str(path_ied)),
    modules=[make_wavelet(), make_detector(), make_inhibitor(),
             make_trigger(n_pulses=3, backoff_s=5.0)],
    config=make_pipeline_config(),
)
dets_3_inh = pipe_3_inh.run_offline()
sw_dets_3 = get_detections(dets_3_inh)
stims_3_inh = get_stims(dets_3_inh)

seqs_inh = defaultdict(list)
for e in stims_3_inh:
    seqs_inh[e.metadata['detection_time']].append(e)

complete = sum(1 for s in seqs_inh.values() if len(s) == 3)
incomplete = sum(1 for s in seqs_inh.values() if 0 < len(s) < 3)
no_stims = len(sw_dets_3) - len(seqs_inh)

print(f'3-pulse + IED inhibition:')
print(f'  {len(sw_dets_3)} detections')
print(f'  {complete} complete (all 3 stims)')
print(f'  {incomplete} partial (some cancelled)')
print(f'  {no_stims} fully cancelled')
print(f'  {len(stims_3_inh)} total stim events')

---
## 5. Detection Report